# PARC2026 EDA と contract check

この notebook は既定で読み取り専用です。データセットが無い場合は旧パスへフォールバックせず、明示的に停止します。

In [ ]:
from pathlib import Path
import os, json, hashlib
import yaml

ROOT = Path.cwd()
if not (ROOT / 'contracts/official_policy_contract.yaml').exists():
    raise RuntimeError('PARC2026_final のルートで実行してください')
contract = yaml.safe_load((ROOT / 'contracts/official_policy_contract.yaml').read_text())
config = yaml.safe_load((ROOT / 'configs/pi05_lora.yaml').read_text())
dataset_root = Path(os.path.expanduser(os.path.expandvars(config['dataset']['root'])))
print(json.dumps({'contract_version': contract['version'], 'dataset_root': str(dataset_root), 'exists': dataset_root.exists()}, ensure_ascii=False, indent=2))
if not dataset_root.exists():
    raise FileNotFoundError(f'dataset root がありません: {dataset_root}')

## 次の読み取り専用チェック

データセットの loader/schema は Omnicampus 配布物に依存するため、配布 root が確認できた後に公式 loader を使って追加する。少なくとも task/episode/frame 数、camera key、画像 dtype/range、state/action shape/range、NaN/Inf、episode 境界、train/validation 重複を記録する。長さ異常と action spike は自動除外せず、人工確認リストにする。

In [ ]:
# 配布データセットに対する manifest の保存は意図的に明示実行にする。
# ここでは旧 revision や本機の別パスへ戻らないことだけを確認する。
assert config['dataset']['revision'] == 'provided-by-Omnicampus'
assert config['dataset']['manifest_sha256'] == '' or len(config['dataset']['manifest_sha256']) == 64
print('contract と config の読み取り専用検査: OK')